In [ ]:
from urbanomy.methods.agent import init_llm
import pandas as pd

llm = init_llm("deepseek/deepseek-v4-flash")

In [ ]:
print(llm.get_name)

In [ ]:

import geopandas as gpd

basline_blocks = gpd.read_file('/data/lhs_icc_sample_50.geojson')
basline_blocks.head()

In [ ]:
feature_cols = [
'residential','business','recreation','industrial','transport','special',
'agriculture','land_use','share','footprint_area','build_floor_area',
'living_area','non_living_area','population','site_area','fsi','gsi',
'mxi','l','morphotype','area_accessibility'
]
cat_features = ['land_use', 'morphotype']
numeric_feats = [c for c in feature_cols if c not in cat_features]
basline_blocks["id"] = basline_blocks.index
basline_blocks['residential'] = basline_blocks['residential'].astype('float64')

In [ ]:
blocks_clean = basline_blocks

In [ ]:
prompts = ['Evaluate the presented scenario based on the criterion of social inclusiveness and low displacement risk.Determine whether the scenario preserves the social and functional diversity of the area, access to everyday services, and opportunities for vulnerable user groups.Assign a score from 0 to 1. Give only score for answer',
           'Evaluate the presented scenario based on the criterion of strategic relevance for the long-term development of the city and district.Consider compactness, functional mix, the balance of housing, jobs, and services, and the area`s ability to adapt to future changes. Assign a score from 0 to 1. Give only score for answer',
           'Evaluate the presented scenario based on its ability to create conditions for active urban life.Consider functional mix, the potential for active ground floors, pedestrian accessibility, public spaces, and diverse everyday use scenarios throughout the day and evening.Assign a score from 0 to 1. Give only score for answer',
           'Evaluate the presented scenario based on the criterion of aligning the interests of various land users and minimizing conflict risk.Analyze potential conflicts between residents, pedestrians, drivers, business owners, visitors, employees, seniors, and families with children.Assign a score from 0 to 1. Give only score for answer',
           'Evaluate the presented scenario based on the criterion of environmental and infrastructure sustainability.Consider building density, transport and infrastructure load, resource efficiency, environmental consequences, and the possibility of phased adaptation of the area.Assign a score from 0 to 1. Give only score for answer']

In [ ]:

from tqdm.auto import tqdm
tasks = []
for index, row in tqdm(
    blocks_clean.iterrows(),
    total=len(blocks_clean),
    desc="Оценка кварталов"
):
    # безопасно получить id квартала
    block_id = int(row['id']) if 'id' in row.index else int(index)

    # короткий контекст для промпта — берём все поля
    ctx_keys = numeric_feats+cat_features
    ctx = {}

    for k in ctx_keys:
        if k in row.index and pd.notnull(row[k]):
            value = row[k]

            if isinstance(value, (int, float)):
                ctx[k] = float(value)
            else:
                ctx[k] = str(value)

    for prompt in prompts:
        task = f"{prompt}\n\nКонтекст района (id={block_id}): {ctx}"
        tasks.append(task)

In [ ]:
tasks

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


MODEL = 'deepseek/deepseek-v4-flash'
llm = init_llm(MODEL)


def invoke(problem, run_id):
    try:
        return {
            'problem': problem,
            'run_id': run_id,
            'model': MODEL,
            'solution': llm.invoke(problem).content
        }
    except Exception as e:
        return {
            'problem': problem,
            'run_id': run_id,
            'model': MODEL,
            'solution': f'ERROR: {e}'
        }


results = []

max_workers = 20

# каждый prompt запускаем 5 раз
jobs = [
    (problem, run_id)
    for problem in tasks
    for run_id in range(5)
]


with ThreadPoolExecutor(max_workers=max_workers) as executor:

    futures = [
        executor.submit(invoke, problem, run_id)
        for problem, run_id in jobs
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="DeepSeek evaluation"
    ):
        results.append(future.result())

In [ ]:
results

In [ ]:
import json

def read_json(file_path : str) -> list | dict:
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)
    
def write_json(data : dict, file_path : str):
    with open(file_path, "w") as f:
        json.dump(data, f)

In [ ]:
write_json(results, 'data/results.json')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from fp2mp_eval import FP2MPEval


N_JUDGES = 5

fp2mp = FP2MPEval(
    model='deepseek/deepseek-v4-flash',
    n_judges=N_JUDGES
)


def evaluate_result(result):
    try:
        problem = result['problem']
        solution = result['solution']

        evals = fp2mp.evaluate_case(
            (problem, solution),
            max_workers=N_JUDGES
        )

        result['evaluations'] = [
            e.model_dump()
            for e in evals
        ]

    except Exception as e:
        result['evaluations'] = [
            {
                "error": str(e)
            }
        ]

    return result


max_workers = 10  # количество параллельных кейсов


evaluated_results = []

with ThreadPoolExecutor(max_workers=max_workers) as executor:

    futures = [
        executor.submit(evaluate_result, result)
        for result in results
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="FP2MP evaluation"
    ):
        evaluated_results.append(future.result())

In [ ]:
evaluated_results.shape

In [ ]:
df = pd.DataFrame(evaluated_results)

df['problem'].unique()

In [ ]:
for p in df['problem'].unique():

    print(p)

In [ ]:
write_json(results, 'data/eval_.json')

In [ ]:
results = evaluated_results

In [ ]:
results

In [ ]:
import pandas as pd
import pingouin as pg


if isinstance(results, dict):
    results = [results]


rows = []

for r in results:
    try:
        score = float(r['solution'])

        rows.append({
            'indicator': r['problem'],
            'judge': r['run_id'],
            'score': score
        })

    except Exception:
        continue


solutions_df = pd.DataFrame(rows)


print(solutions_df.head())
print("valid solutions:", len(solutions_df))


icc_df = pg.intraclass_corr(
    data=solutions_df,
    targets='indicator',
    raters='judge',
    ratings='score',
    nan_policy='omit'
)


# сохраняем ICC в словарь
icc = {}

for _, row in icc_df.iterrows():
    icc_type = row['Type']
    icc_value = row['ICC']
    icc[icc_type] = icc_value


# записываем поле icc в results
for r in results:
    r['icc'] = icc


icc_df

In [ ]:
import pandas as pd


data = []

for result in results:
    problem = result['problem']
    model = result['model']

    if 'icc' not in result:
        continue

    icc = result['icc']

    for icc_type, icc_value in icc.items():
        data.append({
            'problem': problem,
            'model': model,
            'icc_type': icc_type,
            'icc_value': icc_value
        })


data_df = pd.DataFrame(data)

print(data_df.head())


# среднее и стандартное отклонение ICC по типам
icc_summary = (
    data_df
    .groupby(['icc_type'])
    .agg(
        mean=("icc_value", "mean"),
        std=("icc_value", "std"),
    )
    .transpose()
)


icc_summary

In [ ]:
data = []

for result in results:
    problem = result['problem']
    model = result['model']
    if 'icc' not in result: continue
    icc = result['icc']
    for icc_type, icc_value in icc.items():
        data.append({
            'problem': problem,
            'model': model,
            'icc_type': icc_type,
            'icc_value': icc_value
        })

data_df = pd.DataFrame(data)
data_df.head()

In [ ]:
data_df.groupby(['icc_type']).agg(
        mean=("icc_value", "mean"),
        std=("icc_value", "std"),
    ).transpose()